In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd
import numpy as np

In [4]:
# Load CSVs
train_df = pd.read_csv('train_labels.csv')
test_df = pd.read_csv('test_labels.csv')

# Optional: Convert numeric class to string label for clarity
label_map = {
    0: 'Raw_Banana',
    1: 'Raw_Mango',
    2: 'Ripe_Banana',
    3: 'Ripe_Mango'
}
train_df['class'] = train_df['class'].map(label_map)
test_df['class'] = test_df['class'].map(label_map)

# Define image generator (no split)
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='data/train/images',
    x_col='filename',
    y_col='class',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

test_gen = datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='data/test/images',
    x_col='filename',
    y_col='class',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 3999 validated image filenames belonging to 4 classes.
Found 1000 validated image filenames belonging to 4 classes.


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPool2D, Flatten, Dense, BatchNormalization, Dropout

In [10]:
model = Sequential()

model.add(Input(shape=(224, 224, 3)))
model.add(Conv2D(32, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(2))
model.add(Dropout(0.25))

model.add(Conv2D(64, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(2))
model.add(Dropout(0.25))

model.add(Conv2D(128, 3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(2))
model.add(Dropout(0.3))

model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [21]:
model.fit(train_gen, epochs=30, validation_data=test_gen)

Epoch 1/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 423s 3s/step - accuracy: 0.9326 - loss: 0.2134 - val_accuracy: 0.7420 - val_loss: 0.8939
Epoch 2/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 439s 3s/step - accuracy: 0.9241 - loss: 0.2582 - val_accuracy: 0.8630 - val_loss: 1.9083
Epoch 3/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 420s 3s/step - accuracy: 0.9281 - loss: 0.1921 - val_accuracy: 0.8780 - val_loss: 1.2506
Epoch 4/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 444s 3s/step - accuracy: 0.9338 - loss: 0.1705 - val_accuracy: 0.8730 - val_loss: 0.5305
Epoch 5/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 420s 3s/step - accuracy: 0.9503 - loss: 0.1564 - val_accuracy: 0.8160 - val_loss: 1.0806
Epoch 6/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 422s 3s/step - accuracy: 0.9511 - loss: 0.1265 - val_accuracy: 0.8310 - val_loss: 0.7023
Epoch 7/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 439s 3s/step - accuracy: 0.9467 - loss: 0.1516 - val_accuracy: 0.8260 - val_loss: 1.2591
Epoch 8/30
125/125 ━━━━━━━━━━━━━━━━━━━━ 419s 3s/step - accuracy: 0.9257 - loss: 0.2919 - val_accu

In [2]:
import numpy as np
from tensorflow.keras.preprocessing import image
def predict_img(image_path):
    test_image = image.load_img(path=image_path, target_size=(224,224))
    test_image = image.img_to_array(test_image)
    test_image = np.expand_dims(test_image, axis=0)
    class_names = ['Raw_Banana', 'Raw_Mango', 'Ripe_Banana','Ripe_Mango']
    result = model.predict(test_image)
    predicted_class = class_names[np.argmax(result)]
    confidence = np.max(result)

    print(f"Predicted class: {predicted_class} ({confidence:.2f} confidence)")

In [1]:
predict_img('single_prediction/mango_1_u.jpeg')

NameError: name 'predict_img' is not defined

In [24]:
model.save('ripeness_classification_cnn.keras')